# Visualizing GNN & CP-SAT Training Logs
This notebook visualizes the self-supervised pre-training, supervised 5-Fold cross-validation, AI prediction uncertainty, Pareto frontier, Cumulative Risk CDF, and Resource Leveling Profile for the GLPO Logistics Project.

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.ticker as ticker

# Set professional plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 14,
    "font.family": "serif"
})

# Paths
checkpoints_dir = r"E:\University\Year 3 - 3\DA3\ai_pipeline\models\moi\checkpoints"
image_dir = r"E:\University\Year 3 - 3\DA3\docs\baocao2\image"
os.makedirs(image_dir, exist_ok=True)

# Colors
COLOR_TRAIN = "#1e3a8a"
COLOR_VAL = "#b91c1c"
COLOR_ACCENT = "#d97706"

## 1. Self-Supervised Pretraining History
Plots train and validation loss/MAE/RMSE convergence over epochs for the GNN HGT Masked Autoencoder.

In [ ]:
pretrain_path = os.path.join(checkpoints_dir, "pretrain_history.json")
if os.path.exists(pretrain_path):
    with open(pretrain_path, "r", encoding="utf-8") as f:
        pretrain_data = json.load(f)
    
    epochs = [x["epoch"] for x in pretrain_data]
    train_loss = [x["train_loss"] for x in pretrain_data]
    val_loss = [x["val_loss"] for x in pretrain_data]
    val_mae = [x["val_mae"] for x in pretrain_data]
    val_rmse = [x["val_rmse"] for x in pretrain_data]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
    
    ax1.plot(epochs, train_loss, label="Hu\u1ea3n luy\u1ec7n (Train Loss)", color=COLOR_TRAIN, linewidth=2)
    ax1.plot(epochs, val_loss, label="Ki\u1ec3m th\u1eed (Val Loss)", color=COLOR_VAL, linewidth=2, linestyle="--")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("H\u00e0m m\u1ea5t m\u00e1t (NLL Loss)")
    ax1.set_title("Qu\u00e1 tr\u00ecnh h\u1ed9i t\u1ee5 NLL Loss c\u1ee7a HGT MAE")
    ax1.legend()
    ax1.grid(True, linestyle=":", alpha=0.6)
    
    ax2.plot(epochs, val_mae, label="MAE Ki\u1ec3m th\u1eed", color=COLOR_ACCENT, linewidth=2)
    ax2.plot(epochs, val_rmse, label="RMSE Ki\u1ec3m th\u1eed", color="#7c3aed", linewidth=2, linestyle="-.")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Ch\u1ec9 s\u1ed1 Sai s\u1ed1 (H\u1ec7 s\u1ed1 Th\u1eddi l\u00b0\u1ee3ng)")
    ax2.set_title("Sai s\u1ed1 MAE v\u00e0 RMSE trong Pre-training")
    ax2.legend()
    ax2.grid(True, linestyle=":", alpha=0.6)
    
    plt.tight_layout()
    pretrain_out = os.path.join(image_dir, "pretrain_loss.png")
    plt.savefig(pretrain_out, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"Successfully saved to: {pretrain_out}")

## 2. Supervised Fine-Tuning 5-Fold Cross Validation
Plots validation $R^2$ and MSE/Loss convergence over epochs for all 5 folds to demonstrate GNN stability.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
fold_colors = ["#3b82f6", "#ef4444", "#10b981", "#f59e0b", "#6366f1"]

for fold in range(1, 6):
    ft_path = os.path.join(checkpoints_dir, f"finetune_history_fold_{fold}.json")
    if os.path.exists(ft_path):
        with open(ft_path, "r", encoding="utf-8") as f:
            ft_data = json.load(f)
        
        epochs = [x["epoch"] for x in ft_data]
        val_r2 = [x["val_r2"] for x in ft_data]
        val_mse = [x["val_mse"] for x in ft_data]
        
        ax1.plot(epochs, val_r2, label=f"Fold {fold}", color=fold_colors[fold-1], linewidth=1.5)
        ax2.plot(epochs, val_mse, label=f"Fold {fold}", color=fold_colors[fold-1], linewidth=1.5)

ax1.set_xlabel("Epoch")
ax1.set_ylabel("H\u1ec7 s\u1ed1 X\u00e1c \u0111\u1ecbnh R^2 Ki\u1ec3m th\u1eed")
ax1.set_title("\u0110\u01b0\u1eddng cong h\u1ed9i t\u1ee5 R^2 qua 5-Fold CV")
ax1.set_ylim(-0.5, 1.0)
ax1.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax1.legend(loc="lower right")
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.set_xlabel("Epoch")
ax2.set_ylabel("Sai s\u1ed1 B\u00ecnh ph\u01b0\u01a1ng Trung b\u00ecnh (MSE)")
ax2.set_title("Qu\u00e1 tr\u00ecnh h\u1ed9i t\u1ee5 MSE qua 5-Fold CV")
ax2.set_ylim(0, 0.006)
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
epoch_out = os.path.join(image_dir, "epoch.jpg")
plt.savefig(epoch_out, bbox_inches="tight", dpi=300)
plt.show()
print(f"Successfully saved to: {epoch_out}")

## 3. AI Prediction Correlation & Uncertainty
Plots expected delay vs prediction uncertainty to analyze heteroscedastic AI predictions.

In [ ]:
pareto_path = os.path.join(checkpoints_dir, "C2012-04_pareto_results.json")
if os.path.exists(pareto_path):
    with open(pareto_path, "r", encoding="utf-8") as f:
        pareto_data = json.load(f)
    
    if "ai_predictions" in pareto_data:
        preds = pareto_data["ai_predictions"]
        expected_delay = [x["expected_delay"] for x in preds.values()]
        uncertainty_sigma = [x["uncertainty_sigma"] for x in preds.values()]
        
        fig, ax = plt.subplots(figsize=(8, 5.5), dpi=300)
        sns.regplot(x=expected_delay, y=uncertainty_sigma, ax=ax, color="#7c3aed",
                    scatter_kws={"s": 60, "alpha": 0.7, "edgecolors": "black"},
                    line_kws={"color": "red", "lw": 2})
        
        ax.set_xlabel("D\u1ef1 b\u00e1o \u0110\u1ed9 tr\u1ec5 K\u1ef3 v\u1ecdng (expected_delay - gi\u1edd)", fontweight="bold")
        ax.set_ylabel("\u0110\u1ed9 b\u1ea5t \u0111\u1ecbnh D\u1ef1 b\u00e1o (uncertainty_sigma - gi\u1edd)", fontweight="bold")
        ax.set_title("Ph\u00e2n t\u00edch m\u1ed1i t\u01b0\u01a1ng quan gi\u1eefa \u0110\u1ed9 tr\u1ec5 v\u00e0 \u0110\u1ed9 b\u1ea5t \u0111\u1ecbnh c\u1ee7a AI", fontsize=12, fontweight="bold", pad=15)
        ax.grid(True, linestyle=":", alpha=0.6)
        
        scatter_out = os.path.join(image_dir, "hinh3_7_regression_scatter.png")
        plt.savefig(scatter_out, bbox_inches="tight", dpi=300)
        plt.show()
        print(f"Successfully saved to: {scatter_out}")

## 4. CP-SAT Pareto Frontier
Generates the multi-objective Pareto Frontier comparing Makespan (Days) and Total Cost, using Risk percentage to color-code the solutions.

In [ ]:
if os.path.exists(pareto_path):
    with open(pareto_path, "r", encoding="utf-8") as f:
        pareto_data = json.load(f)
    
    if "pareto_options" in pareto_data:
        options = pareto_data["pareto_options"]
        options = sorted(options, key=lambda x: x["makespan_days"])
        
        makespan = [x["makespan_days"] for x in options]
        cost = [x["total_cost"] for x in options]
        risk = [x["risk_pct"] * 100 for x in options]
        names = [x["option_name"] for x in options]
        
        fig, ax = plt.subplots(figsize=(9.5, 6.5), dpi=300)
        
        ax.plot(makespan, cost, color="#94a3b8", linestyle="-", linewidth=2, zorder=1)
        sc = ax.scatter(makespan, cost, c=risk, cmap="plasma", s=180, edgecolors="#1e293b", linewidth=1.5, zorder=2)
        
        for i, (m, c, name) in enumerate(zip(makespan, cost, names)):
            viet_name = name.replace("Option", "P").replace("Ph\u01b0\u01a1ng \xe1n [", "P").replace("]", "")
            ax.annotate(viet_name, (m, c), textcoords="offset points", xytext=(0,12), ha='center',
                        fontsize=10, fontweight="bold", color="#1e293b",
                        bbox=dict(boxstyle="round,pad=0.2", fc="yellow", alpha=0.6, ec="orange"))
            
        ax.set_xlabel("Th\u1eddi gian ho\u00e0n th\u00e0nh d\u1ef1 \u00e1n Makespan (ng\u00e0y)", fontweight="bold", labelpad=10)
        ax.set_ylabel("T\u1ed5ng chi ph\u00ed d\u1ef1 \u00e1n (USD)", fontweight="bold", labelpad=10)
        ax.set_title("\u0110\u01b0\u1eddng cong bi\u00ean Pareto Frontier d\u1ef1 \u00e1n C2012-04", fontsize=13, fontweight="bold", pad=15)
        
        formatter = ticker.FormatStrFormatter('$%1.2f')
        ax.yaxis.set_major_formatter(formatter)
        
        cbar = fig.colorbar(sc, ax=ax)
        cbar.set_label("X\u00e1c su\u1ea5t tr\u1ec5 ti\u1ebfn \u0111\u1ed9 Monte Carlo (%)", fontweight="bold", labelpad=10)
        
        ax.grid(True, linestyle=":", alpha=0.6)
        plt.tight_layout()
        pareto_out = os.path.join(image_dir, "pareto_frontier.png")
        plt.savefig(pareto_out, bbox_inches="tight", dpi=300)
        plt.show()
        print(f"Successfully saved to: {pareto_out}")

## 5. Cumulative Delay Risk CDF
Plots Cumulative Distribution Function of Monte Carlo scheduling simulations.

In [ ]:
if os.path.exists(pareto_path):
    with open(pareto_path, "r", encoding="utf-8") as f:
        pareto_data = json.load(f)
    
    if "monte_carlo_cpm" in pareto_data:
        mc = pareto_data["monte_carlo_cpm"]
        mean = mc["makespan_mean"]
        std = mc["makespan_std"]
        p50 = mc["p50"]
        p80 = mc["p80"]
        p95 = mc["p95"]
        
        np.random.seed(42)
        samples = np.random.normal(mean, std, 10000)
        sorted_samples = np.sort(samples)
        y = np.arange(1, len(sorted_samples) + 1) / len(sorted_samples)
        
        fig, ax = plt.subplots(figsize=(8, 5.5), dpi=300)
        ax.plot(sorted_samples, y * 100, color="#2563eb", linewidth=2.5, label="CDF")
        
        percentiles = [50, 80, 95]
        values = [p50, p80, p95]
        colors = ["#10b981", "#f59e0b", "#ef4444"]
        
        for pct, val, col in zip(percentiles, values, colors):
            ax.axvline(val, color=col, linestyle="--", linewidth=1.2)
            ax.axhline(pct, color=col, linestyle="--", linewidth=1.2)
            ax.plot(val, pct, marker="o", color=col, markersize=8)
            ax.text(val + 10, pct - 4, f"P{pct} = {val:.1f}h", color=col, fontweight="bold")
            
        ax.set_xlabel("Th\u1eddi gian ho\u00e0n th\u00e0nh d\u1ef1 \u00e1n (gi\u1edd)", fontweight="bold")
        ax.set_ylabel("X\u00e1c su\u1ea5t t\u00edch lu\u1ef5y ho\u00e0n th\u00e0nh (%)", fontweight="bold")
        ax.set_title("Ph\u00e2n ph\u1ed1i t\u00edch lu\u1ef5y CDF r\u1ee7i ro ti\u1ebfn \u0111\u1ed9 Monte Carlo", fontsize=12, fontweight="bold", pad=15)
        ax.grid(True, linestyle=":", alpha=0.6)
        
        cdf_out = os.path.join(image_dir, "cdf_risk.png")
        plt.savefig(cdf_out, bbox_inches="tight", dpi=300)
        plt.show()
        print(f"Successfully saved to: {cdf_out}")

## 6. Resource Leveling Profile (Before vs After Optimization)
Plots labor resource requirements over time for Option 1 (before leveling) and Option 7 (after leveling).

In [ ]:
if os.path.exists(pareto_path):
    with open(pareto_path, "r", encoding="utf-8") as f:
        pareto_data = json.load(f)
    
    if "pareto_options" in pareto_data:
        options = pareto_data["pareto_options"]
        opt_before = options[0]
        opt_after = options[-1]
        
        def calculate_load(op):
            schedule = op.get("tasks_schedule", {})
            makespan_hours = int(op.get("makespan_hours", 0))
            load = np.zeros(makespan_hours)
            for tid, t in schedule.items():
                s = int(t["start_hours"])
                f = int(t["finish_hours"])
                w = int(t["assigned_workers"]) + int(t.get("extra_workers", 0))
                load[s:f] += w
            return load
        
        load_before = calculate_load(opt_before)
        load_after = calculate_load(opt_after)
        
        # Plot Before
        fig, ax = plt.subplots(figsize=(10, 4.5), dpi=300)
        ax.fill_between(range(len(load_before)), load_before, color="#fca5a5", alpha=0.6, label="Nhu c\u1ea7u nh\u00e2n l\u1ef1c")
        ax.plot(range(len(load_before)), load_before, color="#dc2626", linewidth=1.5)
        ax.axhline(13.0, color="purple", linestyle="--", linewidth=1.5, label="Peak Limit = 13")
        ax.set_xlabel("Th\u1eddi gian th\u1ef1c hi\u1ec7n d\u1ef1 \u00e1n (gi\u1edd)")
        ax.set_ylabel("S\u1ed1 l\u01b0\u1ee3ng nh\u00e2n l\u1ef1c huy \u0111\u1ed9ng (ng\u01b0\u1eddi)")
        ax.set_title("Bi\u1ec3u \u0111\u1ed3 ph\u00e2n b\u1ed5 t\u00e0i nguy\u00ean nh\u00e2n l\u1ef1c TR\u01af\u1edaC khi t\u1ed1i \u01b0u h\u00f3a", fontsize=11, fontweight="bold")
        ax.legend(loc="upper right")
        ax.set_ylim(0, 15)
        ax.grid(True, linestyle=":", alpha=0.6)
        res_before_out = os.path.join(image_dir, "hinh3_3_resource_before.png")
        plt.savefig(res_before_out, bbox_inches="tight", dpi=300)
        plt.show()
        print(f"Successfully saved to: {res_before_out}")
        
        # Plot After
        fig, ax = plt.subplots(figsize=(10, 4.5), dpi=300)
        ax.fill_between(range(len(load_after)), load_after, color="#86efac", alpha=0.6, label="Nhu c\u1ea7u nh\u00e2n l\u1ef1c")
        ax.plot(range(len(load_after)), load_after, color="#16a34a", linewidth=1.5)
        ax.axhline(11.0, color="purple", linestyle="--", linewidth=1.5, label="Peak Limit = 11")
        ax.set_xlabel("Th\u1eddi gian th\u1ef1c hi\u1ec7n d\u1ef1 \u00e1n (gi\u1edd)")
        ax.set_ylabel("S\u1ed1 l\u01b0\u1ee3ng nh\u00e2n l\u1ef1c huy \u0111\u1ed9ng (ng\u01b0\u1eddi)")
        ax.set_title("Bi\u1ec3u \u0111\u1ed3 ph\u00e2n b\u1ed5 t\u00e0i nguy\u00ean nh\u00e2n l\u1ef1c SAU khi t\u1ed1i \u01b0u h\u00f3a", fontsize=11, fontweight="bold")
        ax.legend(loc="upper right")
        ax.set_ylim(0, 15)
        ax.grid(True, linestyle=":", alpha=0.6)
        res_after_out = os.path.join(image_dir, "hinh3_4_resource_after.png")
        plt.savefig(res_after_out, bbox_inches="tight", dpi=300)
        plt.show()
        print(f"Successfully saved to: {res_after_out}")